# GSB 5544 — Topic 5.1: JSON and APIs — SOLUTION
*Data that is nested instead of rectangular, how to flatten it, and how to ask a web server for it*

## The next 15 minutes

| | Question | Where it lands in PA 5.1 |
|---|---|---|
| **1. Read** | What does JSON look like, and how do I reach a value inside it? | everything |
| **2. Flatten** | How does nested data become a data frame? (`pd.json_normalize`) | Shows data 1 – 4, Tasty 5 |
| **3. Request** | How do I ask an API for JSON? (`requests.get`, endpoints, parameters) | TVMaze 1, Tasty 1 |
| **4. Repeat** | How do I make many requests politely? (loop, `extend`, `time.sleep`) | TVMaze 2, Tasty 3 |

Until now every data set arrived as a rectangle: rows and columns. Data from the web usually arrives
**hierarchical** — a show *has* a network, and *has many* episodes, each of which *has* a title. JSON is the
format that holds that shape.

In [1]:
import pandas as pd
import requests
import time

---
## 1. JSON is dictionaries and lists, nested

Once loaded into Python, JSON is made of things you already know:

| JSON | Python | Reach inside with |
|---|---|---|
| object `{ "name": "Girls" }` | **dictionary** | a **key**: `d["name"]` |
| array `[ ..., ... ]` | **list** | a **position**: `x[0]` |
| string, number, `true`/`false`, `null` | `str`, `int`/`float`, `True`/`False`, `None` | — |

Two shows, written by hand. Each show is a dictionary; `network` is a dictionary *inside* it;
`episodes` is a *list of dictionaries* inside it.

In [2]:
shows = [
    {"name": "Derry Girls", "network": {"name": "Channel 4", "country": "GB"},
     "episodes": [{"title": "Episode 1", "runtime": 30}, {"title": "Episode 2", "runtime": 30}]},
    {"name": "Bomb Girls", "network": {"name": "Global", "country": "CA"},
     "episodes": [{"title": "Jumping Tracks", "runtime": 60}]},
]
type(shows), len(shows), shows[0].keys()

(list, 2, dict_keys(['name', 'network', 'episodes']))

To reach a value, **walk down one level at a time**: list → position, dictionary → key.

In [3]:
shows[0]["network"]["name"]          # first show -> its network -> the network's name

'Channel 4'

In [4]:
shows[1]["episodes"][0]["title"]     # the title of Bomb Girls' first episode

'Jumping Tracks'

✅ (a) Is `shows` a list or a dictionary? What about `shows[0]`? And `shows[0]["episodes"]`?
(b) Why does `shows["name"]` raise an error?

**Answer:** (a) `shows` is a **list** (of two shows); `shows[0]` is a **dictionary** (one show); `shows[0]["episodes"]` is
a **list** again (of episode dictionaries). (b) `shows` is a list, and lists take positions, not keys — you
must pick a show first: `shows[0]["name"]`. When you are lost in real JSON, ask `type(...)` and, for a
dictionary, `.keys()`.

---
## 2. Flattening: `pd.json_normalize`

**One row per show.** Nested *dictionaries* become dotted column names. Nested *lists* cannot fit in one cell
of a rectangle, so they are left as lists.

In [5]:
df_shows = pd.json_normalize(shows)
df_shows

,name,episodes,network.name,network.country
0,Derry Girls,"[{'title': 'Episode 1', 'runtime': 30}, {'titl...",Channel 4,GB
1,Bomb Girls,"[{'title': 'Jumping Tracks', 'runtime': 60}]",Global,CA


**One row per episode.** To open up a list, name it as the `record_path`. Each element of that list becomes
a row. Use `meta` to carry down fields from the parent so you know which show each episode belongs to.

In [6]:
df_episodes = pd.json_normalize(shows,
                                record_path="episodes",     # the list to unpack: one row per element
                                meta=["name"])              # parent fields to keep on each row
df_episodes

,title,runtime,name
0,Episode 1,30,Derry Girls
1,Episode 2,30,Derry Girls
2,Jumping Tracks,60,Bomb Girls


✅ (a) `df_shows` has ____ rows and `df_episodes` has ____ rows — what decides each number? (b) Which data
frame would you use to count shows per network? Which to find the average episode runtime per show?

**Answer:** (a) 2 rows (one per **show**) and 3 rows (one per **episode**: 2 + 1). The `record_path` decides what a row
*is*. (b) Shows per network → `df_shows` (`df_shows["network.name"].value_counts()`); runtime per show →
`df_episodes` (`groupby("name")["runtime"].mean()`). **Choosing the unit of the row is the key decision** —
after that it is ordinary pandas. If the list is two levels down, give the path as a list:
`record_path=["seasons", "episodes"]`.

---
## 3. Asking an API for JSON

An **API** is a web address built for programs rather than people: you send a request, it sends back JSON.
A request has three parts:

| Part | Example | |
|---|---|---|
| **base URL** | `https://api.tvmaze.com` | the service |
| **endpoint** | `/search/shows` | *which* kind of data (listed in the API's documentation) |
| **parameters** | `q=golden girls` | the details of *this* question; joined to the URL after a `?` |

`requests.get` sends it. Pass parameters as a dictionary and `requests` builds the `?q=...` part for you.

In [7]:
response = requests.get("https://api.tvmaze.com/search/shows", params={"q": "golden girls"})
response.url, response.status_code

('https://api.tvmaze.com/search/shows?q=golden+girls', 200)

`status_code` 200 means success (404 = no such page, 401/403 = not authorised, 429 = slow down). Always
look at it before trusting the data. `.json()` converts the reply into Python lists and dictionaries:

In [8]:
results = response.json()
type(results), len(results), results[0].keys()

(list, 4, dict_keys(['score', 'show']))

In [9]:
pd.json_normalize(results)[["score", "show.id", "show.name", "show.premiered", "show.network.name"]]

,score,show.id,show.name,show.premiered,show.network.name
0,1.136332,5451,Golden Girls,2012-08-25,RTL4
1,1.027853,71642,Golden Girls,2011-05-26,Channel 10
2,1.004472,722,The Golden Girls,1985-09-14,NBC
3,0.451813,57291,The Holden Girls: Mandy & Myrtle,2021-09-07,E4


✅ (a) Each element of `results` has two keys, `score` and `show`. Why do the column names start with
`show.`? (b) What would you change to search for *"powerpuff"* instead?

**Answer:** (a) The show's details are a dictionary nested under the key `show`, and `json_normalize` names nested
fields `parent.child` — hence `show.name`, and two levels down, `show.network.name`. (b) Only the parameter:
`params={"q": "powerpuff"}`. The base URL and endpoint stay the same — that is the whole point of an endpoint.

---
## 4. Many requests: loop, collect, pause

Some questions need one request per item (the cast of *each* show) or per page of results. The pattern:

1. start an **empty list**;
2. **loop**, making one request per item;
3. add each reply to the list with **`.extend(...)`** (adds the *elements* of the reply — `.append` would add
   the whole reply as one element);
4. **`time.sleep(...)`** between requests, so the server does not block you;
5. flatten **once, after** the loop — in a *separate cell*, so you never re-request just to fix your pandas.

In [10]:
show_ids = [722, 33320]              # The Golden Girls, Derry Girls

cast = []
for show_id in show_ids:
    response = requests.get(f"https://api.tvmaze.com/shows/{show_id}/cast")
    cast.extend(response.json())
    time.sleep(0.5)                  # be polite: half a second between requests

len(cast)

16

In [11]:
pd.json_normalize(cast)[["person.name", "character.name"]].head()

,person.name,character.name
0,Bea Arthur,Dorothy Zbornak
1,Betty White,Rose Nylund
2,Rue McClanahan,Blanche Devereaux
3,Estelle Getty,Sophia Petrillo
4,Saoirse-Monica Jackson,Erin Quinn


✅ After this loop you cannot tell which show each cast member came from. Why not — and what one line inside
the loop would fix it?

**Answer:** The cast endpoint returns people and characters but not the show, and `extend` pours every reply into one
list. Tag each record before adding it — for example
`members = response.json()`, then `for m in members: m["show_id"] = show_id`, then `cast.extend(members)`.
You will need exactly this in PA 5.1 (TVMaze part 2).

---
## 5. APIs that need a key

Some APIs (the Tasty API in the PA) only answer registered users. You sign up, receive an **API key**, and
send it in the request's **headers** — extra information that travels with the request but is not part of
the URL:

```python
headers = {"X-RapidAPI-Key": "your-key-here", "X-RapidAPI-Host": "tasty.p.rapidapi.com"}
response = requests.get(url, headers=headers, params={"q": "daikon"})
```

A key is a password: it counts *your* requests against *your* quota. Do not post it, and do not leave it in a
notebook you share or push to GitHub.

## The four lines to keep

| | |
|---|---|
| **Read** | JSON = dictionaries (`["key"]`) and lists (`[0]`), nested; when lost, `type()` and `.keys()` |
| **Flatten** | `pd.json_normalize(data)` = one row per top-level item; `record_path=` opens a list, `meta=` keeps the parent's fields |
| **Request** | `requests.get(url, params={...})` → check `.status_code` → `.json()` |
| **Repeat** | empty list → loop → `.extend()` → `time.sleep()`; request in one cell, process in another |

PA 5.1 is on the course site: [https://gato365.github.io/gsb5544_instructor_learn_prep/](https://gato365.github.io/gsb5544_instructor_learn_prep/).